In [ ]:
import rfc8785

canonical = rfc8785.dumps({"b": 2, "a": 1})
assert isinstance(canonical, bytes)
assert canonical == b'{"a":1,"b":2}'  # keys sorted, no whitespace

# Insertion order never leaks into the canonical form.
assert rfc8785.dumps({"a": 1, "b": 2}) == rfc8785.dumps({"b": 2, "a": 1})
print(canonical)

In [ ]:
import rfc8785

# ECMAScript-style number formatting (shortest round-trip form).
assert rfc8785.dumps([10.0, 1.0, -0.0]) == b"[10,1,0]"
out = rfc8785.dumps({"numbers": [1e30, 4.5, 0.002, 1e-27]})
assert out == b'{"numbers":[1e+30,4.5,0.002,1e-27]}'

# Property keys sort by UTF-16 code units: "A" < "a" < "\u00e9" < "\u20ac".
out = rfc8785.dumps({"\u20ac": "euro", "a": 1, "\u00e9": 2, "A": 3})
assert out == '{"A":3,"a":1,"\u00e9":2,"\u20ac":"euro"}'.encode("utf-8")
print(out.decode("utf-8"))

In [ ]:
# Dump to a binary IO + round-trip invariant with the stdlib json parser
import io, json
import rfc8785

value = {"k": [1, 2], "nested": {"s": "text", "flag": True, "none": None}}
buf = io.BytesIO()
rfc8785.dump(value, buf)
serialized = buf.getvalue()
assert serialized == rfc8785.dumps(value)

# Canonicalization is idempotent: parse + re-canonicalize is a fixed point.
assert rfc8785.dumps(json.loads(serialized)) == serialized
print(serialized.decode("utf-8"))


In [ ]:
# Documented failure domain: the exception hierarchy fires on non-JCS values
import rfc8785

assert issubclass(rfc8785.IntegerDomainError, rfc8785.CanonicalizationError)
assert issubclass(rfc8785.FloatDomainError, rfc8785.CanonicalizationError)

for bad, expected in [
    (float("nan"), rfc8785.FloatDomainError),   # NaN is unrepresentable in JCS
    (float("inf"), rfc8785.FloatDomainError),   # Infinity is unrepresentable in JCS
    (2**53, rfc8785.IntegerDomainError),        # beyond the IEEE-754 safe-integer range
    (object(), rfc8785.CanonicalizationError),  # unsupported type
]:
    try:
        rfc8785.dumps(bad)
        raise AssertionError(f"{bad!r} unexpectedly serialized")
    except expected as e:
        print(f"OK {expected.__name__}: {e}")

# The largest safe integer still serializes.
assert rfc8785.dumps(2**53 - 1) == b"9007199254740991"


In [ ]:
# The flagship use case: canonical bytes as a stable digest input
# (this is why the sigstore/DSSE stack depends on rfc8785)
import hashlib
import rfc8785

payload_a = {"spec": {"version": 1, "name": "artifact"}, "kind": "attestation"}
payload_b = {"kind": "attestation", "spec": {"name": "artifact", "version": 1}}

digest_a = hashlib.sha256(rfc8785.dumps(payload_a)).hexdigest()
digest_b = hashlib.sha256(rfc8785.dumps(payload_b)).hexdigest()
assert digest_a == digest_b  # key order cannot perturb the digest
print("stable digest:", digest_a)
